# VLM 逻辑监工：多模态模型集成
## Floor Plan 3D 重建质检流水线

```
build_3d_model()
      │
      ▼
render_inspection_views()   ← 自动截取俯视图 + 侧视图
      │
      ▼
vlm_inspect()               ← 发送截图给 Claude claude-sonnet-4-20250514
      │
      ├─ score >= threshold → preview_service.approve() → 进入生产库
      └─ score <  threshold → persistence_service.reject(keep=True) → 待人工复核
```

### 对应论文扩展
论文 Section 2.5 只做几何重建，不做语义验证。  
本文件在 `build_3d_model` 结束后增加一个 **VLM 质检层**，检查：
- 门是否开在窗户上
- 墙体是否悬空（无支撑）
- 房间是否闭合（是否存在缺口）
- 门窗比例是否符合建筑规范

### 文件依赖
```
postprocess_config.py  ← 阈值配置
reconstruct_3d.py      ← build_3d_model, match_openings_to_walls
post_service.py        ← run_postprocess
preview_service.py     ← generate_preview
persistence_service.py ← approve / reject
```

## 0. 环境配置

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, sys, json, time, io, base64, uuid
from pathlib import Path
from typing import List, Optional, Tuple
from dataclasses import dataclass, field

import cv2
import numpy as np
from PIL import Image
import anthropic          # pip install anthropic

# 3D 渲染
try:
    import trimesh
    import pyrender         # pip install pyrender
    HAS_RENDER = True
except ImportError:
    HAS_RENDER = False
    print('pyrender 未安装，将用 trimesh 离线截图替代')
    print('安装: pip install pyrender')

# 项目模块（假设已在同目录）
sys.path.insert(0, '.')          # 把项目根目录加入 path
from postprocess_config import (
    OUTPUT_DIR, STAGING_DIR, STAGING_CFG,
    VECT_CFG, INF_CFG
)
from reconstruct_3d  import build_3d_model, match_openings_to_walls
from post_service    import run_postprocess, run_inference
from preview_service import generate_preview
from persistence_service import approve, reject, list_pending

# ── 路径 ──
VLM_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'vlm_inspection')
os.makedirs(VLM_OUTPUT_DIR, exist_ok=True)

# ── Anthropic 客户端 ──
# 优先读环境变量 ANTHROPIC_API_KEY，也可以直接赋值
client = anthropic.Anthropic()   # 自动读取 ANTHROPIC_API_KEY

print('✓ 环境配置完成')
print(f'  VLM_OUTPUT_DIR : {VLM_OUTPUT_DIR}')
print(f'  HAS_RENDER     : {HAS_RENDER}')

## 1. VLM 质检配置

In [ ]:
@dataclass
class VLMInspectorConfig:
    """
    VLM 逻辑监工配置
    所有阈值和提示词集中在这里，方便 A/B 测试不同 prompt 策略
    """
    # ── 模型 ──
    model:          str   = 'claude-sonnet-4-20250514'
    max_tokens:     int   = 1024

    # ── 质检阈值（0-10 分，低于此分数 → 待人工复核）──
    auto_approve_threshold: float = 7.0   # >= 7 分自动通过
    auto_reject_threshold:  float = 3.0   # <= 3 分直接拒绝
    # 3 < score < 7 → 标记为待人工复核（human_review）

    # ── 截图配置 ──
    render_width:   int   = 800
    render_height:  int   = 600
    views: List[str] = field(default_factory=lambda: ['top', 'front', 'side'])

    # ── 检查项权重（影响最终得分计算）──
    check_weights: dict = field(default_factory=lambda: {
        'door_on_window':   3.0,   # 门开在窗上：严重错误
        'wall_floating':    3.0,   # 墙体悬空：严重错误
        'room_not_closed':  2.0,   # 房间不闭合：中等错误
        'proportion_wrong': 2.0,   # 比例不合理：轻微错误
    })

    # ── 系统提示词（角色设定）──
    system_prompt: str = (
        '你是一位专业的建筑审图工程师，专注于检查 3D 户型图的结构逻辑正确性。'
        '你的判断必须严格、客观，基于建筑规范和几何逻辑。'
        '你的输出必须是合法的 JSON，不要包含任何 Markdown 标记或额外说明。'
    )

    # ── 用户提示词模板（{view_desc} 会被替换）──
    user_prompt_template: str = (
        '请检查这张 3D 户型图的{view_desc}，'
        '对以下四项进行独立评估，每项给出 true/false 和简短说明：\n'
        '1. door_on_window: 是否存在门开在窗户上的情况\n'
        '2. wall_floating: 是否存在墙体悬空（墙体两端无连接或无支撑）\n'
        '3. room_not_closed: 是否存在房间边界不闭合（出现明显缺口）\n'
        '4. proportion_wrong: 门/窗的尺寸比例是否明显不合建筑规范\n\n'
        '返回格式（严格 JSON，不要 Markdown）：\n'
        '{\n'
        '  "door_on_window":   {"issue": false, "detail": "说明"},\n'
        '  "wall_floating":    {"issue": false, "detail": "说明"},\n'
        '  "room_not_closed":  {"issue": false, "detail": "说明"},\n'
        '  "proportion_wrong": {"issue": false, "detail": "说明"},\n'
        '  "overall_score":    8,\n'
        '  "summary":          "整体质量说明（1-2句）"\n'
        '}'
    )


VLM_CFG = VLMInspectorConfig()
print('✓ VLM 质检配置完成')
print(f'  模型              : {VLM_CFG.model}')
print(f'  自动通过阈值      : {VLM_CFG.auto_approve_threshold}')
print(f'  自动拒绝阈值      : {VLM_CFG.auto_reject_threshold}')
print(f'  截图视角          : {VLM_CFG.views}')

## 2. 3D 模型截图：俯视图 + 侧视图

In [ ]:
def render_with_pyrender(
    scene:  'trimesh.Scene',
    view:   str,
    width:  int = 800,
    height: int = 600,
) -> np.ndarray:
    """
    用 pyrender 离屏渲染 3D 场景
    view: 'top' | 'front' | 'side'
    返回 RGB ndarray (H, W, 3)
    """
    import pyrender as pr
    import trimesh

    # 把 trimesh.Scene 转成 pyrender.Scene
    pr_scene = pr.Scene.from_trimesh_scene(scene, ambient_light=[0.5,0.5,0.5])

    # 计算场景包围盒，用于摆放相机
    bounds  = scene.bounds          # [[xmin,ymin,zmin],[xmax,ymax,zmax]]
    center  = scene.centroid
    extent  = np.max(scene.extents) * 1.5

    # 相机位置
    cam_poses = {
        'top':   _look_at(center + [0, 0, extent], center, up=[0, 1, 0]),
        'front': _look_at(center + [0, -extent, extent*0.5], center, up=[0, 0, 1]),
        'side':  _look_at(center + [extent, 0, extent*0.5], center, up=[0, 0, 1]),
    }
    cam_pose = cam_poses.get(view, cam_poses['top'])

    camera = pr.PerspectiveCamera(yfov=np.pi / 3.0, aspectRatio=width/height)
    pr_scene.add(camera, pose=cam_pose)

    # 添加平行光
    light = pr.DirectionalLight(color=[1,1,1], intensity=3.0)
    pr_scene.add(light, pose=cam_pose)

    # 离屏渲染
    renderer = pr.OffscreenRenderer(width, height)
    color, _ = renderer.render(pr_scene)
    renderer.delete()
    return color   # (H, W, 3) uint8 RGB


def render_with_trimesh_offscreen(
    scene:  'trimesh.Scene',
    view:   str,
    width:  int = 800,
    height: int = 600,
) -> np.ndarray:
    """
    trimesh 内置离屏渲染（不需要 pyrender，但效果较简单）
    用于没有 pyrender 的环境
    """
    import trimesh

    bounds = scene.bounds
    center = scene.centroid
    extent = np.max(scene.extents) * 1.5

    # 调整相机视角
    if view == 'top':
        angles = [-np.pi/2, 0, 0]
    elif view == 'front':
        angles = [0, 0, 0]
    else:  # side
        angles = [0, np.pi/2, 0]

    # 用 scene.save_image（需要安装 xvfb 或 osmesa）
    try:
        scene.camera_transform = trimesh.transformations.euler_matrix(*angles)
        png_bytes = scene.save_image(resolution=[width, height], visible=False)
        img = np.array(Image.open(io.BytesIO(png_bytes)))
        return img[:, :, :3]  # 去掉 alpha
    except Exception:
        # 兜底：返回白色占位图
        placeholder = np.ones((height, width, 3), dtype=np.uint8) * 240
        cv2.putText(placeholder, f'render failed ({view})',
                    (20, height//2), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                    (100, 100, 100), 2)
        return placeholder


def _look_at(eye, target, up):
    """构建 look-at 变换矩阵（4x4 camera pose）"""
    eye    = np.array(eye,    dtype=float)
    target = np.array(target, dtype=float)
    up     = np.array(up,     dtype=float)

    forward = eye - target
    forward /= (np.linalg.norm(forward) + 1e-8)

    right = np.cross(up, forward)
    right_norm = np.linalg.norm(right)
    if right_norm < 1e-8:
        up = np.array([1, 0, 0], dtype=float)
        right = np.cross(up, forward)
        right_norm = np.linalg.norm(right)
    right /= right_norm

    up_actual = np.cross(forward, right)

    pose = np.eye(4)
    pose[:3, 0] = right
    pose[:3, 1] = up_actual
    pose[:3, 2] = forward
    pose[:3, 3] = eye
    return pose


def render_inspection_views(
    scene:    'trimesh.Scene',
    task_id:  str,
    save_dir: str,
    cfg:      VLMInspectorConfig = VLM_CFG,
) -> dict:
    """
    为 3D 场景生成多角度质检截图
    优先用 pyrender，不可用时 fallback 到 trimesh 内置渲染

    返回：
    {
        'top':   {'path': str, 'image': ndarray},
        'front': {'path': str, 'image': ndarray},
        'side':  {'path': str, 'image': ndarray},
    }
    """
    os.makedirs(save_dir, exist_ok=True)
    views_out = {}

    for view in cfg.views:
        print(f'  渲染 {view} 视图...', end=' ')
        try:
            if HAS_RENDER:
                img = render_with_pyrender(scene, view, cfg.render_width, cfg.render_height)
            else:
                img = render_with_trimesh_offscreen(scene, view, cfg.render_width, cfg.render_height)
        except Exception as e:
            print(f'渲染失败({e}), 使用占位图')
            img = np.ones((cfg.render_height, cfg.render_width, 3), dtype=np.uint8) * 230
            cv2.putText(img, f'View: {view}  (render error)',
                        (20, cfg.render_height//2), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (80,80,80), 2)

        path = os.path.join(save_dir, f'{task_id}_{view}.png')
        cv2.imwrite(path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        views_out[view] = {'path': path, 'image': img}
        print(f'✓ {path}')

    return views_out


print('✓ 截图工具函数定义完成')
print('  render_inspection_views(scene, task_id, save_dir) → 多视角截图 dict')

## 3. VLM 质检核心函数

In [ ]:
def _image_to_base64(image: np.ndarray) -> str:
    """把 RGB ndarray 编码成 base64 字符串（PNG 格式）"""
    pil_img = Image.fromarray(image.astype(np.uint8))
    buf = io.BytesIO()
    pil_img.save(buf, format='PNG')
    return base64.standard_b64encode(buf.getvalue()).decode('utf-8')


def _parse_vlm_response(text: str) -> dict:
    """
    解析 VLM 返回的 JSON
    宽容解析：去除 Markdown 代码块标记，处理尾随逗号等常见问题
    """
    # 去掉 markdown 代码块
    text = text.strip()
    if text.startswith('```'):
        lines = text.split('\n')
        text  = '\n'.join(lines[1:-1]) if lines[-1] == '```' else '\n'.join(lines[1:])
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # 兜底：返回中性结果，不影响流程
        print(f'  [警告] VLM 响应解析失败，原始内容:\n{text[:300]}')
        return {
            'door_on_window':   {'issue': False, 'detail': 'parse_error'},
            'wall_floating':    {'issue': False, 'detail': 'parse_error'},
            'room_not_closed':  {'issue': False, 'detail': 'parse_error'},
            'proportion_wrong': {'issue': False, 'detail': 'parse_error'},
            'overall_score':    5,
            'summary':          'VLM 响应解析失败，使用中性默认分',
        }


def vlm_inspect_single_view(
    image:     np.ndarray,
    view_name: str,
    cfg:       VLMInspectorConfig = VLM_CFG,
) -> dict:
    """
    对单张截图调用 VLM 质检
    使用 Claude Sonnet 4 的视觉能力
    返回解析后的质检结果 dict
    """
    view_desc_map = {
        'top':   '俯视图（从上方正视）',
        'front': '正视图（从正面看）',
        'side':  '侧视图（从侧面看）',
    }
    view_desc = view_desc_map.get(view_name, view_name)
    img_b64   = _image_to_base64(image)

    message = client.messages.create(
        model      = cfg.model,
        max_tokens = cfg.max_tokens,
        system     = cfg.system_prompt,
        messages   = [
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': {
                            'type':       'base64',
                            'media_type': 'image/png',
                            'data':       img_b64,
                        },
                    },
                    {
                        'type': 'text',
                        'text': cfg.user_prompt_template.format(view_desc=view_desc),
                    },
                ],
            }
        ],
    )

    raw_text = message.content[0].text
    result   = _parse_vlm_response(raw_text)
    result['view']          = view_name
    result['input_tokens']  = message.usage.input_tokens
    result['output_tokens'] = message.usage.output_tokens
    return result


def vlm_inspect(
    views:   dict,
    task_id: str,
    cfg:     VLMInspectorConfig = VLM_CFG,
) -> dict:
    """
    对所有视角截图执行 VLM 质检，聚合结果

    views: render_inspection_views() 返回的 dict

    返回聚合报告：
    {
        task_id:          str,
        view_results:     {view: result_dict},
        aggregated_score: float,       # 0-10
        decision:         str,         # 'auto_approve' | 'human_review' | 'auto_reject'
        issues_found:     list,        # 发现的问题列表
        total_tokens:     int,
        rejection_reason: str | None,  # 对应 persistence_service 的 REJECTION_REASONS
    }
    """
    print(f'\n[{task_id}] 开始 VLM 质检，共 {len(views)} 个视角...')
    view_results  = {}
    total_tokens  = 0
    all_scores    = []
    all_issues    = []

    for view_name, view_data in views.items():
        print(f'  [{view_name}] 发送给 VLM...', end=' ', flush=True)
        t0     = time.time()
        result = vlm_inspect_single_view(view_data['image'], view_name, cfg)
        elapsed = round(time.time() - t0, 1)
        print(f'✓ score={result.get("overall_score", "?")}  ({elapsed}s)')

        view_results[view_name] = result
        total_tokens += result.get('input_tokens', 0) + result.get('output_tokens', 0)

        score = result.get('overall_score', 5)
        if isinstance(score, (int, float)):
            all_scores.append(float(score))

        # 收集发现的问题
        for check_name in cfg.check_weights:
            check = result.get(check_name, {})
            if isinstance(check, dict) and check.get('issue', False):
                all_issues.append({
                    'view':   view_name,
                    'type':   check_name,
                    'detail': check.get('detail', ''),
                })

    # ── 聚合得分（取各视角最低分，保守策略）──
    aggregated_score = min(all_scores) if all_scores else 5.0

    # ── 决策 ──
    if aggregated_score >= cfg.auto_approve_threshold:
        decision = 'auto_approve'
    elif aggregated_score <= cfg.auto_reject_threshold:
        decision = 'auto_reject'
    else:
        decision = 'human_review'

    # ── 推断最主要的拒绝原因（用于数据闭环）──
    rejection_reason = None
    if all_issues:
        # 按严重程度排序（权重高的问题优先）
        sorted_issues  = sorted(
            all_issues,
            key=lambda x: cfg.check_weights.get(x['type'], 0),
            reverse=True,
        )
        issue_type_map = {
            'door_on_window':   'door_offset',
            'wall_floating':    'geometry_invalid',
            'room_not_closed':  'wall_missing',
            'proportion_wrong': 'scale_wrong',
        }
        rejection_reason = issue_type_map.get(sorted_issues[0]['type'], 'other')

    report = {
        'task_id':          task_id,
        'view_results':     view_results,
        'aggregated_score': round(aggregated_score, 2),
        'decision':         decision,
        'issues_found':     all_issues,
        'total_tokens':     total_tokens,
        'rejection_reason': rejection_reason,
        'timestamp':        time.strftime('%Y-%m-%dT%H:%M:%SZ'),
    }

    # 打印摘要
    decision_emoji = {'auto_approve': '✅', 'human_review': '🔍', 'auto_reject': '❌'}
    print(f'\n  聚合得分  : {aggregated_score:.1f} / 10')
    print(f'  决策      : {decision_emoji.get(decision, "?")} {decision}')
    print(f'  问题数量  : {len(all_issues)}')
    print(f'  Token 消耗: {total_tokens}')
    if all_issues:
        print('  发现的问题:')
        for iss in all_issues:
            print(f'    [{iss["view"]}] {iss["type"]}: {iss["detail"]}')

    return report


print('✓ VLM 质检函数定义完成')
print('  vlm_inspect(views, task_id) → 聚合质检报告')

## 4. 完整 VLM 质检流水线

In [ ]:
def run_vlm_inspection_pipeline(
    image_path:     str,
    vlm_cfg:        VLMInspectorConfig = VLM_CFG,
    upload_to_gcs:  bool = False,
    dry_run:        bool = False,
) -> dict:
    """
    完整 VLM 质检流水线

    流程：
      1. run_postprocess()         → 推理 + 矢量化 + 3D 重建
      2. generate_preview()        → 写入 staging
      3. render_inspection_views() → 多角度截图
      4. vlm_inspect()             → VLM 多视角质检
      5. 根据决策调用 approve / reject / human_review 标记

    参数：
      dry_run: True 时不实际调用 VLM API（用于测试流程，返回 mock 结果）

    返回完整 pipeline 报告 dict
    """
    task_id  = str(uuid.uuid4())[:8]
    name     = Path(image_path).stem
    t_start  = time.time()

    print(f'\n{"="*60}')
    print(f'任务 [{task_id}]  图片: {name}')
    print(f'{"="*60}')

    # ── Step 1: 后处理（推理 + 矢量化 + 3D 重建）──
    print('\nStep 1: 后处理流程...')
    postprocess_result = run_postprocess(
        image_path = image_path,
        output_dir = os.path.join(STAGING_DIR, task_id),
        export_3d  = True,
        task_id    = task_id,
    )
    glb_path = postprocess_result.get('glb_path')
    print(f'  walls={postprocess_result["stats"]["n_walls"]}  '
          f'doors={postprocess_result["stats"]["n_doors"]}  '
          f'windows={postprocess_result["stats"]["n_windows"]}')

    # ── Step 2: 从 GLB 文件重新加载场景（或直接使用返回的场景对象）──
    print('\nStep 2: 加载 3D 场景...')
    scene = None
    if glb_path and os.path.exists(glb_path):
        try:
            import trimesh
            scene = trimesh.load(glb_path)
            print(f'  ✓ 3D 场景加载成功: {glb_path}')
        except Exception as e:
            print(f'  ✗ 3D 场景加载失败: {e}')
    else:
        print('  3D 场景不存在，跳过 VLM 质检')

    # ── Step 3: 渲染质检截图 ──
    views = {}
    if scene is not None:
        print('\nStep 3: 渲染质检截图...')
        view_dir = os.path.join(VLM_OUTPUT_DIR, task_id)
        views    = render_inspection_views(scene, task_id, view_dir, vlm_cfg)
    else:
        # 没有 3D 场景时，用 wall mask 的俯视图做简单替代
        print('\nStep 3: 无 3D 场景，使用 wall mask 作为替代视图...')
        views = _fallback_views_from_postprocess(postprocess_result, task_id, vlm_cfg)

    # ── Step 4: VLM 质检 ──
    print('\nStep 4: VLM 质检...')
    if dry_run:
        print('  [dry_run 模式] 跳过实际 VLM 调用，返回 mock 结果')
        vlm_report = _mock_vlm_report(task_id, vlm_cfg)
    else:
        vlm_report = vlm_inspect(views, task_id, vlm_cfg)

    # ── Step 5: 根据决策执行 approve / reject / human_review ──
    print('\nStep 5: 执行决策...')
    decision = vlm_report['decision']

    if decision == 'auto_approve':
        action_result = approve(task_id, upload_to_gcs=upload_to_gcs)
        print(f'  ✅ 自动通过  → {action_result.get("archive_dir", "")}')

    elif decision == 'auto_reject':
        reason = vlm_report.get('rejection_reason') or 'geometry_invalid'
        action_result = reject(task_id, reason=reason, note='VLM 自动拒绝', keep_file=False)
        print(f'  ❌ 自动拒绝  reason={reason}')

    else:  # human_review
        # 保留文件并写入 human_review 标记
        action_result = _mark_for_human_review(task_id, vlm_report)
        print(f'  🔍 标记为待人工复核  score={vlm_report["aggregated_score"]}')

    # ── 保存完整报告 ──
    elapsed = round(time.time() - t_start, 2)
    pipeline_report = {
        'task_id':            task_id,
        'image_path':         image_path,
        'postprocess_stats':  postprocess_result['stats'],
        'vlm_report':         vlm_report,
        'decision':           decision,
        'action_result':      action_result,
        'total_elapsed':      elapsed,
    }

    report_path = os.path.join(VLM_OUTPUT_DIR, f'{task_id}_pipeline_report.json')
    with open(report_path, 'w', encoding='utf-8') as f:
        json.dump(pipeline_report, f, indent=2, ensure_ascii=False, default=str)

    print(f'\n完整报告已保存: {report_path}')
    print(f'总耗时: {elapsed}s')
    return pipeline_report


# ──────────────────────────────────────────
# 辅助函数
# ──────────────────────────────────────────

def _fallback_views_from_postprocess(
    postprocess_result: dict,
    task_id: str,
    cfg: VLMInspectorConfig,
) -> dict:
    """
    无 3D 场景时，把矢量化 overlay 图作为替代的俯视图
    VLM 仍可检查 2D 层面的逻辑问题
    """
    vis_files = list(Path(STAGING_DIR, task_id).glob('*_vis.png'))
    views = {}
    if vis_files:
        img = cv2.cvtColor(cv2.imread(str(vis_files[0])), cv2.COLOR_BGR2RGB)
        img_path = os.path.join(VLM_OUTPUT_DIR, task_id, f'{task_id}_top.png')
        os.makedirs(os.path.dirname(img_path), exist_ok=True)
        cv2.imwrite(img_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        views['top'] = {'path': img_path, 'image': img}
    return views


def _mark_for_human_review(task_id: str, vlm_report: dict) -> dict:
    """在 staging 目录写入 human_review 标记文件"""
    staging_task_dir = os.path.join(STAGING_CFG.staging_dir, task_id)
    if not os.path.exists(staging_task_dir):
        return {'task_id': task_id, 'status': 'human_review', 'warning': 'staging_dir_missing'}

    mark = {
        'status':           'human_review',
        'vlm_score':        vlm_report['aggregated_score'],
        'issues':           vlm_report['issues_found'],
        'rejection_reason': vlm_report.get('rejection_reason'),
        'timestamp':        time.strftime('%Y-%m-%dT%H:%M:%SZ'),
    }
    mark_path = os.path.join(staging_task_dir, 'human_review.json')
    with open(mark_path, 'w') as f:
        json.dump(mark, f, indent=2)
    return {'task_id': task_id, 'status': 'human_review', 'mark_path': mark_path}


def _mock_vlm_report(task_id: str, cfg: VLMInspectorConfig) -> dict:
    """dry_run 模式：返回 mock 质检结果（不消耗 API 配额）"""
    mock_result = {
        'door_on_window':   {'issue': False, 'detail': '[mock] 未发现门开在窗上'},
        'wall_floating':    {'issue': False, 'detail': '[mock] 墙体连接正常'},
        'room_not_closed':  {'issue': True,  'detail': '[mock] 发现一处疑似缺口'},
        'proportion_wrong': {'issue': False, 'detail': '[mock] 比例正常'},
        'overall_score':    6,
        'summary':          '[mock] 存在疑似墙体缺口，建议人工复核',
        'view':             'mock',
        'input_tokens':     0,
        'output_tokens':    0,
    }
    return {
        'task_id':          task_id,
        'view_results':     {'mock': mock_result},
        'aggregated_score': 6.0,
        'decision':         'human_review',
        'issues_found':     [{'view': 'mock', 'type': 'room_not_closed', 'detail': '[mock] 疑似缺口'}],
        'total_tokens':     0,
        'rejection_reason': 'wall_missing',
        'timestamp':        time.strftime('%Y-%m-%dT%H:%M:%SZ'),
    }


print('✓ 完整 VLM 质检流水线定义完成')
print('  run_vlm_inspection_pipeline(image_path, dry_run=True) → 完整报告')

## 5. 单张图片测试（dry_run 模式，不消耗 API）

In [ ]:
# ══════════════════════════════════════════════════════════════
# 测试 1：dry_run 模式（验证流程，不调用 VLM API）
# 把 dry_run=False 换成真实调用
# ══════════════════════════════════════════════════════════════

from config import DATA_FOLDER
from numpy import genfromtxt

val_folders  = genfromtxt(DATA_FOLDER + 'val.txt', dtype='str')
test_folder  = val_folders[0].strip('/')
test_img     = os.path.join(DATA_FOLDER, test_folder, 'F1_scaled.png')

print(f'测试图片: {test_img}')
print('模式: dry_run=True（不消耗 API 配额）')
print('如需真实 VLM 调用，将 dry_run=False')

report = run_vlm_inspection_pipeline(
    image_path = test_img,
    dry_run    = True,     # ← 改为 False 使用真实 VLM
)

print(f'\n任务 ID     : {report["task_id"]}')
print(f'VLM 得分    : {report["vlm_report"]["aggregated_score"]}')
print(f'决策        : {report["decision"]}')
print(f'总耗时      : {report["total_elapsed"]}s')

## 6. 真实 VLM 调用（单张图片）

In [ ]:
# ══════════════════════════════════════════════════════════════
# 测试 2：真实 VLM 调用（需要有效的 ANTHROPIC_API_KEY）
# 推荐先用 dry_run 验证流程再开启
# ══════════════════════════════════════════════════════════════

RUN_REAL_VLM = False   # ← 改为 True 开启真实调用

if RUN_REAL_VLM:
    print('开始真实 VLM 质检...')
    report_real = run_vlm_inspection_pipeline(
        image_path = test_img,
        dry_run    = False,
    )

    # 打印各视角质检细节
    print('\n各视角质检细节：')
    for view_name, vr in report_real['vlm_report']['view_results'].items():
        print(f'\n  [{view_name}] 得分={vr.get("overall_score")}  {vr.get("summary", "")}')
        for check in ['door_on_window', 'wall_floating', 'room_not_closed', 'proportion_wrong']:
            c = vr.get(check, {})
            flag = '🚨' if c.get('issue') else '✓'
            print(f'    {flag} {check}: {c.get("detail", "")}')
else:
    print('跳过真实 VLM 调用（RUN_REAL_VLM=False）')
    print('如需运行，将 RUN_REAL_VLM 改为 True')

## 7. 批量 VLM 质检（验证集）

In [ ]:
def batch_vlm_inspect(
    data_folder: str,
    n_samples:   int   = 10,
    dry_run:     bool  = True,
    save_dir:    str   = VLM_OUTPUT_DIR,
) -> dict:
    """
    批量对验证集运行 VLM 质检

    返回汇总统计：
    {
        'auto_approve':  count,
        'human_review':  count,
        'auto_reject':   count,
        'avg_score':     float,
        'total_tokens':  int,
        'reports':       list,
    }
    """
    from numpy import genfromtxt

    val_folders = genfromtxt(data_folder + 'val.txt', dtype='str')
    folders     = val_folders[:n_samples]

    print(f'批量 VLM 质检  n={n_samples}  dry_run={dry_run}')
    print('─' * 50)

    stats   = {'auto_approve': 0, 'human_review': 0, 'auto_reject': 0}
    scores  = []
    tokens  = []
    reports = []

    for i, folder in enumerate(folders):
        folder   = folder.strip('/')
        img_path = os.path.join(data_folder, folder, 'F1_scaled.png')

        if not os.path.exists(img_path):
            print(f'[{i+1}/{n_samples}] 跳过（图片不存在）: {folder}')
            continue

        print(f'[{i+1}/{n_samples}] {folder}')
        try:
            r = run_vlm_inspection_pipeline(img_path, dry_run=dry_run)
            decision = r['decision']
            score    = r['vlm_report']['aggregated_score']
            tok      = r['vlm_report']['total_tokens']

            stats[decision] = stats.get(decision, 0) + 1
            scores.append(score)
            tokens.append(tok)
            reports.append({'folder': folder, 'decision': decision,
                            'score': score, 'tokens': tok})

            print(f'  → {decision}  score={score}  tokens={tok}')

        except Exception as e:
            print(f'  → ERROR: {e}')

    # 汇总
    n = len(scores)
    summary = {
        **stats,
        'total':         n,
        'avg_score':     round(sum(scores)/n, 2) if n else 0,
        'total_tokens':  sum(tokens),
        'reports':       reports,
    }

    print(f'\n{"─"*50}')
    print(f'批量质检完成  共 {n} 个样本')
    print(f'  ✅ auto_approve  : {stats["auto_approve"]}')
    print(f'  🔍 human_review  : {stats["human_review"]}')
    print(f'  ❌ auto_reject   : {stats["auto_reject"]}')
    print(f'  avg_score       : {summary["avg_score"]}')
    print(f'  total_tokens    : {summary["total_tokens"]}')

    # 保存汇总
    path = os.path.join(save_dir, 'batch_vlm_summary.json')
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f'  汇总已保存: {path}')

    return summary


# ── 运行批量质检 ──
RUN_BATCH = False   # ← 改为 True 开始批量

if RUN_BATCH:
    batch_summary = batch_vlm_inspect(
        data_folder = DATA_FOLDER,
        n_samples   = 10,
        dry_run     = True,    # ← 改为 False 使用真实 VLM
    )
else:
    print('跳过批量质检（RUN_BATCH=False）')
    print('如需运行，将 RUN_BATCH 改为 True')

## 8. 质检结果可视化

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def visualize_vlm_report(
    pipeline_report: dict,
    save_path: str = None,
):
    """
    可视化单次 VLM 质检报告
    展示：各视角截图 + 质检细节 + 决策结果
    """
    vlm   = pipeline_report['vlm_report']
    views = vlm.get('view_results', {})
    n_views = len(views)

    fig = plt.figure(figsize=(6 * max(n_views, 1) + 4, 10))
    gs  = gridspec.GridSpec(2, max(n_views, 1) + 1, figure=fig)

    # ── 上半部分：各视角截图 ──
    task_dir = os.path.join(VLM_OUTPUT_DIR, pipeline_report['task_id'])
    for col, (view_name, vr) in enumerate(views.items()):
        ax = fig.add_subplot(gs[0, col])
        # 尝试读取已保存的截图
        img_path = os.path.join(task_dir, f'{pipeline_report["task_id"]}_{view_name}.png')
        if os.path.exists(img_path):
            img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, f'{view_name}\n(no image)',
                    ha='center', va='center', transform=ax.transAxes)
        score = vr.get('overall_score', '?')
        ax.set_title(f'{view_name} view  score={score}', fontsize=11)
        ax.axis('off')

    # ── 上半部分最右：雷达图（4 个检查项）──
    ax_radar = fig.add_subplot(gs[0, -1])
    _draw_issue_summary(ax_radar, vlm, pipeline_report)

    # ── 下半部分：质检细节文本 ──
    ax_txt = fig.add_subplot(gs[1, :])
    ax_txt.axis('off')
    lines = []
    decision_sym = {'auto_approve': '✅', 'human_review': '🔍', 'auto_reject': '❌'}
    lines.append(f'任务 {pipeline_report["task_id"]}  '
                 f'决策: {decision_sym.get(vlm["decision"],"?")} {vlm["decision"]}  '
                 f'聚合得分: {vlm["aggregated_score"]:.1f}/10  '
                 f'Token: {vlm["total_tokens"]}')
    lines.append('')
    if vlm['issues_found']:
        lines.append('发现的问题：')
        for iss in vlm['issues_found']:
            lines.append(f'  [{iss["view"]}] {iss["type"]}: {iss["detail"]}')
    else:
        lines.append('未发现结构逻辑问题 ✓')

    for vn, vr in views.items():
        lines.append(f'\n  [{vn}] {vr.get("summary", "")}')

    ax_txt.text(0.02, 0.95, '\n'.join(lines),
                transform=ax_txt.transAxes,
                fontsize=9, va='top', family='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.suptitle(f'VLM 质检报告  {pipeline_report["task_id"]}', fontsize=14)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        print(f'报告图已保存: {save_path}')
    plt.show()


def _draw_issue_summary(ax, vlm: dict, pipeline_report: dict):
    """绘制 4 个检查项的问题频率柱状图（简单替代雷达图）"""
    checks  = ['door_on_window', 'wall_floating', 'room_not_closed', 'proportion_wrong']
    labels  = ['门/窗重叠', '墙体悬空', '房间不闭合', '比例异常']
    # 统计各视角中每类问题的出现次数
    counts  = [0] * len(checks)
    for vr in vlm.get('view_results', {}).values():
        for i, c in enumerate(checks):
            if vr.get(c, {}).get('issue', False):
                counts[i] += 1
    colors = ['red' if c > 0 else 'lightgreen' for c in counts]
    ax.bar(labels, counts, color=colors, edgecolor='gray')
    ax.set_title('问题检测结果', fontsize=10)
    ax.set_ylabel('触发视角数')
    ax.set_ylim(0, max(max(counts) + 1, 3))
    ax.tick_params(axis='x', labelsize=8)
    for i, (label, cnt) in enumerate(zip(labels, counts)):
        ax.text(i, cnt + 0.05, str(cnt), ha='center', fontsize=9)


# 可视化刚才的 dry_run 报告
visualize_vlm_report(
    pipeline_report = report,
    save_path       = os.path.join(VLM_OUTPUT_DIR, f'{report["task_id"]}_report.png'),
)

## 9. 导出为生产 .py 文件

In [ ]:
# ══════════════════════════════════════════════════════════════
# 这个 Cell 告诉你如何把本 Notebook 的逻辑
# 拆分到项目的 .py 结构中
# ══════════════════════════════════════════════════════════════

EXPORT_GUIDE = """
生产化拆分建议
══════════════════════════════════════

vlm_inspector.py          ← 本 Notebook 的算法层
  VLMInspectorConfig      （Cell 1 的配置类）
  render_inspection_views （Cell 2 的截图函数）
  vlm_inspect             （Cell 3 的质检函数）
  run_vlm_inspection_pipeline （Cell 4 的完整流水线）

调用关系：

  post_service.py
    └─ run_postprocess()          ← 现有

  vlm_inspector.py               ← 新增
    ├─ render_inspection_views()  ← 截图
    └─ vlm_inspect()              ← VLM 调用

  preview_service.py
    └─ generate_preview()         ← 在这里插入 VLM 质检
         ├─ run_postprocess()     （已有）
         ├─ vlm_inspect()         （新增调用）
         └─ 根据 decision → approve / reject / human_review

CI/CD 更新：
  requirements.txt 新增：
    anthropic>=0.40.0
    pyrender>=0.1.45   （可选，有 mesa/osmesa 才能用）
    trimesh>=4.0.0
"""

print(EXPORT_GUIDE)

## 10. 查看待审核任务队列

In [ ]:
# ══════════════════════════════════════════════════════════════
# 运维工具：查看当前待人工复核的任务列表
# 配合 persistence_service.py 的 approve/reject 命令使用
# ══════════════════════════════════════════════════════════════

pending = list_pending()
print(f'待审核任务: {len(pending)} 个')
print('─' * 55)
for p in pending:
    print(f'  task_id : {p.get("task_id", "?")}'
          f'  walls={p.get("stats", {}).get("n_walls", "?")}'
          f'  doors={p.get("stats", {}).get("n_doors", "?")}'
          f'  elapsed={p.get("elapsed", "?")}s')
    # 读取 human_review.json 如果存在
    task_dir   = os.path.join(STAGING_DIR, p.get('task_id', ''))
    review_f   = os.path.join(task_dir, 'human_review.json')
    if os.path.exists(review_f):
        with open(review_f) as f:
            rv = json.load(f)
        print(f'    VLM score={rv.get("vlm_score")}  '
              f'issues={len(rv.get("issues", []))}')
    print()

print('─' * 55)
print('通过命令行操作：')
print('  # 通过某个任务')
print('  python persistence_service.py --approve --task_id <id>')
print('  # 拒绝并标记原因')
print('  python persistence_service.py --reject --task_id <id> --reason wall_missing')